In [1]:
import edn_popdyn.paths
from edn_popdyn.paths import (
    POP_MAT_DIRNAME, DATA_ROOT, REP_DRIFT_DATA,
)

# 0.1 — Regenerate spike vector arrays with trial-aligned reaction times

Build per-condition population matrices from single neuron spike vectors and align single-trial reaction times (RT) accoridngly

## 0) Imports

In [2]:
import os
import re
import pickle
import warnings

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from collections import Counter

from pathlib import Path
from edn_popdyn.paths import PROCESSED_DATA_DIR

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

## 1) Configuration

In [3]:
area = 'ALL'

data_dir            = f'{REP_DRIFT_DATA}/'
chronic_data_folder = data_dir + 'full_set/chronic_spikesorted/'
chronic_prepared    = chronic_data_folder + 'prepared_jan_25_2022/'

base_dir_pop_mat    = (f'{DATA_ROOT}/{area}/{POP_MAT_DIRNAME}/'
                     '20msbins/brokenup_AP_CI_4')
trial_events_dir = f'{DATA_ROOT}/{area}/trial_events'
txt_file_path    = f'{DATA_ROOT}/bird_str_rec_str_{area}_list.txt'


POP_MAT_OUT = Path(PROCESSED_DATA_DIR) / 'pop_mats'
overwrite_existing  = False
min_trials_per_unit = 0
nbins               = 60
gaussian_sigma_ms   = 25
n_jobs              = 20

import re as _re_cfg
_ts_re_cfg = _re_cfg.compile(r'(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d+)')

def _detect_sorter(spike_dir, rec_timestamp):

    rec_root = os.path.join(spike_dir, 'trial_aligned_spikes_padding_100ms')
    if not os.path.isdir(rec_root):
        return None
    for sorter in sorted(os.listdir(rec_root)):
        sorter_path = os.path.join(rec_root, sorter)
        if not os.path.isdir(sorter_path):
            continue
        for d in os.listdir(sorter_path):
            m = _ts_re_cfg.search(d)
            if m and m.group(1) == rec_timestamp:
                return sorter
    return None

bird_rec_pairs = []
current_bird = None
with open(txt_file_path) as fh:
    for line in fh:
        line = line.strip()
        if line.startswith('B'):
            current_bird = line
        elif current_bird and line.startswith('exp'):
            bird_rec_pairs.append((current_bird, line))

seen = set()
bird_rec_pairs = [p for p in bird_rec_pairs if not (p in seen or seen.add(p))]

birds_list  = []
no_sorter   = []
no_ts       = []
for bird, exp_name in bird_rec_pairs:
    m = _ts_re_cfg.search(exp_name)
    if m is None:
        no_ts.append((bird, exp_name))
        continue
    rec_timestamp = m.group(1)
    spike_dir     = os.path.join(chronic_prepared, bird) + '/'
    sorter = _detect_sorter(spike_dir, rec_timestamp)
    if sorter is None:
        no_sorter.append((bird, rec_timestamp))
        continue
    birds_list.append((bird, rec_timestamp, sorter))

print(f"area = {area}")
print(f"Loaded {len(bird_rec_pairs)} (bird, rec) pairs from {txt_file_path}")
print(f"Resolved sorter for {len(birds_list)}/{len(bird_rec_pairs)} pairs")
if no_ts:
    print(f"  WARNING: {len(no_ts)} entries had no parseable timestamp")
    for b, e in no_ts[:5]:
        print(f"    {b} / {e}")
if no_sorter:
    print(f"  WARNING: {len(no_sorter)} pairs had no matching sorter directory")
    for b, t in no_sorter[:5]:
        print(f"    {b} / {t}")

area = ALL
Loaded 537 (bird, rec) pairs from /mnt/cube/jugorman/EDNpopdyn/data/bird_str_rec_str_ALL_list.txt
Resolved sorter for 408/537 pairs
    B1188 / 2020-11-03_18-18-57_2250
    B1188 / 2020-11-08_11-52-37_2250
    B1188 / 2020-11-08_20-11-48_2250
    B1188 / 2020-11-09_20-26-08_2250
    B1188 / 2020-11-10_21-23-06_2250


## 2) Path resolvers

In [4]:
_ts_re = re.compile(r'(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d+)')

def find_existing_exp_dir(base_dir_pop_mat, bird_str, rec_name_str):
    bird_path = os.path.join(base_dir_pop_mat, bird_str)
    if not os.path.isdir(bird_path):
        return None
    for d in sorted(os.listdir(bird_path)):
        full = os.path.join(bird_path, d)
        if not os.path.isdir(full):
            continue
        m = _ts_re.search(d)
        if m is not None and m.group(1) == rec_name_str:
            return full
    return None

def find_spike_rec_dir(spikesorting_folder, sorter, rec_name_str):
    sort_dir = os.path.join(spikesorting_folder,
                            'trial_aligned_spikes_padding_100ms', sorter)
    if not os.path.isdir(sort_dir):
        return None
    for d in sorted(os.listdir(sort_dir)):
        full = os.path.join(sort_dir, d)
        if not os.path.isdir(full):
            continue
        m = _ts_re.search(d)
        if m is not None and m.group(1) == rec_name_str:
            return full
    return None

def find_trial_events_path(trial_events_dir, exp_dir_name):
    for suffix in ('_trial_events_full.pickle', '_trial_events.pickle'):
        p = os.path.join(trial_events_dir, exp_dir_name + suffix)
        if os.path.exists(p):
            return p
    return None

## 3) Helpers 

In [5]:
def list_unit_pickles_in_dir(spike_rec_dir, min_trials_per_unit=0):
    if not os.path.isdir(spike_rec_dir):
        return []
    out = []
    for fname in sorted(os.listdir(spike_rec_dir)):
        if not fname.endswith('.pickle'):
            continue
        unit_name = fname[:-len('.pickle')]
        if min_trials_per_unit > 0:
            try:
                df = pd.read_pickle(os.path.join(spike_rec_dir, fname))
                if len(df) < min_trials_per_unit:
                    continue
            except Exception:
                continue
        out.append((unit_name, spike_rec_dir))
    return out

def load_unit_spike_trains_from_dir(unit_recording_ids):
    rows = []
    for unit, spike_rec_dir in tqdm(unit_recording_ids,
                                    desc='unit spike trains', leave=False):
        path = os.path.join(spike_rec_dir, f'{unit}.pickle')
        if not os.path.exists(path):
            continue
        df = pd.read_pickle(path)
        df['recording_id'] = os.path.basename(spike_rec_dir)

        cue_list, interp_list, ip_list = [], [], []
        for stim in df['stim'].values:
            parts = stim.split('_')
            if len(parts) == 3:
                cue_info, interp_info, ip_info = parts
                if '.wav' in ip_info:
                    ip_info = ip_info[:-4]
                cue_list.append(cue_info if len(cue_info) in (2, 3) else 'NaN')
                interp_list.append(interp_info if len(interp_info) == 2 and interp_info.isalpha() else 'NaN')
                ip_list.append(ip_info if len(ip_info) == 3 and ip_info.isdigit() else 'NaN')
            else:
                cue_list.append('NaN'); interp_list.append('NaN'); ip_list.append('NaN')
        df['cue'] = cue_list
        df['interp'] = interp_list
        df['interp_point'] = ip_list
        df = df[(df['cue'] != 'NaN') &
                (df['interp'] != 'NaN') &
                (df['interp_point'] != 'NaN')]
        rows.append(df)
    if not rows:
        return None
    return pd.concat(rows, ignore_index=True)

def calculate_cue_validity(row):
    if row['cue'] in ('CR1', 'CR0'):
        return row['correct_choice'] == 'right'
    if row['cue'] in ('CL1', 'CL0'):
        return row['correct_choice'] == 'left'
    return np.nan

def get_pop_mat_from_trial_aligned_spikes_df(df, nbins, gaussian_sigma_ms=25):
    bin_ms = 1000 / nbins
    sigma  = gaussian_sigma_ms / bin_ms
    pop_mats = []
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
        if row.cue in ('CL1', 'CL0', 'CN', 'CR0', 'CR1'):
            bins = np.linspace(1, 2, nbins + 1)
        else:
            bins = np.linspace(0, 1, nbins + 1)
        hist, _ = np.histogram(row.spike_times, bins, density=False)
        pop_mats.append(gaussian_filter1d(hist.astype('float'), sigma, mode='constant'))
    df['pop_mat'] = pop_mats
    return df

def create_pop_mat_and_rt_aligned(df, key_col='trial_id'):

    sorted_keys  = np.array(sorted(df[key_col].unique()))
    unique_units = np.array(sorted(df['unit'].unique()))
    pop_mat_length  = len(df['pop_mat'].iloc[0])

    rt_per_trial = (df.drop_duplicates(key_col)
                      .set_index(key_col)['rt'])

    grouped = (df.groupby([key_col, 'unit'])['pop_mat'].first().to_dict())

    Z = np.zeros((len(sorted_keys), len(unique_units), pop_mat_length))
    for i, k in enumerate(sorted_keys):
        for j, u in enumerate(unique_units):
            pop_mat = grouped.get((k, u), None)
            if pop_mat is not None:
                Z[i, j, :] = pop_mat

    rt = np.array([rt_per_trial.get(k, np.nan) for k in sorted_keys],
                  dtype=float)
    return Z, rt, sorted_keys

## 4) Trial-events RT loader with frame_begin fallback

`trial_id` is preferred when present (B1170, B1248, B1597). Falls back to
`frame_begin` for sessions without `trial_id` (B1595).

In [6]:
def load_rt_for_recording(exp_dir_name, trial_events_dir):

    path = find_trial_events_path(trial_events_dir, exp_dir_name)
    if path is None:
        return None, None, f"no trial_events file for {exp_dir_name}"
    try:
        te = pd.read_pickle(path)
    except Exception as e:
        return None, None, f"failed to read {os.path.basename(path)}: {e}"
    if len(te) == 0:
        return None, None, f"empty trial_events file: {os.path.basename(path)}"
    if 'rt' not in te.columns:
        return None, None, f"no 'rt' column in {os.path.basename(path)}"
    if 'trial_id' in te.columns and te['trial_id'].notna().any():
        s = te.drop_duplicates('trial_id').set_index('trial_id')['rt']
        return (s, 'trial_id',
                f"loaded {len(te)} rows, joining on trial_id, "
                f"{te['rt'].notna().sum()} finite RTs")
    if 'frame_begin' in te.columns:
        s = te.drop_duplicates('frame_begin').set_index('frame_begin')['rt']
        return (s, 'frame_begin',
                f"loaded {len(te)} rows, joining on frame_begin "
                f"(no trial_id column), {te['rt'].notna().sum()} finite RTs")
    return None, None, f"no joinable column in {os.path.basename(path)}"

def merge_rt_into_all_trials(all_trials, rt_series, join_key):
    all_trials = all_trials.copy()
    if rt_series is not None and join_key in all_trials.columns:
        all_trials['rt'] = all_trials[join_key].map(rt_series)
    else:
        all_trials['rt'] = np.nan
    return all_trials

## 5) Per-session worker

In [7]:
def process_bird_rec_pair(bird_str, rec_name_str, sorter_str):
    out = {'bird': bird_str, 'rec': rec_name_str,
           'rt_status': None, 'join_key': None,
           'n_units': 0, 'n_trials_loaded': 0, 'n_rt_finite': 0,
           'n_keys_written': 0, 'error': None}
    try:
        exp_dir = find_existing_exp_dir(base_dir_pop_mat, bird_str, rec_name_str)
        if exp_dir is None:
            out['error'] = "no existing exp_dir"
            return out
        exp_dir_name = os.path.basename(exp_dir)

        out_dir = POP_MAT_OUT / bird_str / exp_dir_name          # <-- CHANGED (edit 2)
        out_dir.mkdir(parents=True, exist_ok=True)               # <-- CHANGED (edit 2)

        spikesorting_folder = (chronic_prepared +
                               bird_str + '/')
        spike_rec_dir = find_spike_rec_dir(spikesorting_folder, sorter_str,
                                           rec_name_str)
        if spike_rec_dir is None:
            out['error'] = f"no spike_rec_dir for sorter={sorter_str}"
            return out
        unit_recording_ids = list_unit_pickles_in_dir(
            spike_rec_dir, min_trials_per_unit=min_trials_per_unit
        )
        if not unit_recording_ids:
            out['error'] = "no unit pickles in spike_rec_dir"
            return out
        out['n_units'] = len(unit_recording_ids)
        all_trials = load_unit_spike_trains_from_dir(unit_recording_ids)
        if all_trials is None or all_trials.empty:
            out['error'] = "spike loader returned empty"
            return out
        all_trials = all_trials.copy()
        out['n_trials_loaded'] = all_trials['trial_id'].nunique()
        rt_series, join_key, msg = load_rt_for_recording(
            exp_dir_name, trial_events_dir
        )
        out['rt_status'] = msg
        out['join_key']  = join_key
        if rt_series is None:
            join_key = 'trial_id'
            all_trials['rt'] = np.nan
        else:
            all_trials = merge_rt_into_all_trials(all_trials, rt_series, join_key)
        out['n_rt_finite'] = int(
            all_trials.drop_duplicates(join_key)['rt'].notna().sum()
        )
        all_trials['interp_point']   = all_trials['interp_point'].astype(int)
        all_trials['correct_choice'] = np.where(
            all_trials['interp_point'] > 62, 'left', 'right'
        )
        all_trials['cue_validity']   = all_trials.apply(calculate_cue_validity, axis=1)
        all_trials = get_pop_mat_from_trial_aligned_spikes_df(
            all_trials, nbins, gaussian_sigma_ms=gaussian_sigma_ms
        )
        interp_ranges = {
            'sL': (0,   31),
            'wL': (32,  62),
            'wR': (63,  84),
            'sR': (85, 125),
        }
        n_written = 0
        for interp in all_trials['interp'].unique():
            for prefix, (lo, hi) in interp_ranges.items():
                active = all_trials[
                    (all_trials['passive'] == False) &
                    (all_trials['interp'] == interp) &
                    (all_trials['interp_point'].between(lo, hi))
                ]
                conds = {
                    'valid_correct':     ((active['correct'] == True)  & (active['cue_validity'] == True)),
                    'valid_incorrect':   ((active['correct'] == False) & (active['cue_validity'] == True)),
                    'invalid_correct':   ((active['correct'] == True)  & (active['cue_validity'] == False)),
                    'invalid_incorrect': ((active['correct'] == False) & (active['cue_validity'] == False)),
                    'NC_correct':        ((active['correct'] == True)  & (active['cue'] == 'NC')),
                    'NC_incorrect':      ((active['correct'] == False) & (active['cue'] == 'NC')),
                }
                for category, mask in conds.items():
                    subset = active[mask]
                    if subset.empty:
                        continue
                    key = f'{prefix}_{interp}_Z_{category}_active'
                    pop_mat_path = os.path.join(out_dir, f'{key}.pkl')      
                    rt_path   = os.path.join(out_dir, f'{key}_rt.pkl')      
                    if (not overwrite_existing) and os.path.exists(pop_mat_path) and os.path.exists(rt_path):
                        continue
                    Z, rt_arr, _ = create_pop_mat_and_rt_aligned(subset, key_col=join_key)
                    with open(pop_mat_path, 'wb') as f:                    
                        pickle.dump(Z, f)                                   
                    with open(rt_path, 'wb') as f:                          
                        pickle.dump(rt_arr, f)                              
                    n_written += 1
        for interp in all_trials['interp'].unique():
            for prefix, (lo, hi) in interp_ranges.items():
                passive = all_trials[
                    (all_trials['passive'] == True) &
                    (all_trials['interp'] == interp) &
                    (all_trials['interp_point'].between(lo, hi))
                ]
                conds = {
                    'valid':   (passive['cue_validity'] == True),
                    'invalid': (passive['cue_validity'] == False),
                    'NC':      (passive['cue'] == 'NC'),
                }
                for category, mask in conds.items():
                    subset = passive[mask]
                    if subset.empty:
                        continue
                    key = f'{prefix}_{interp}_Z_{category}_passive'
                    pop_mat_path = os.path.join(out_dir, f'{key}.pkl')      
                    rt_path   = os.path.join(out_dir, f'{key}_rt.pkl')      
                    if (not overwrite_existing) and os.path.exists(pop_mat_path) and os.path.exists(rt_path):
                        continue
                    Z, rt_arr, _ = create_pop_mat_and_rt_aligned(subset, key_col=join_key)
                    with open(pop_mat_path, 'wb') as f:                     
                        pickle.dump(Z, f)                                  
                    with open(rt_path, 'wb') as f:                         
                        pickle.dump(rt_arr, f)                             
                    n_written += 1
        out['n_keys_written'] = n_written
        return out
    except Exception as e:
        out['error'] = f"{type(e).__name__}: {e}"
        return out

## 6) Run + summary

In [8]:
results = Parallel(n_jobs=n_jobs)(
    delayed(process_bird_rec_pair)(bird, rec, sorter)
    for bird, rec, sorter in birds_list
)

summary = pd.DataFrame(results)
print("\n=== Per-session summary ===")
cols = ['bird', 'rec', 'n_units', 'n_trials_loaded', 'n_rt_finite',
        'n_keys_written', 'join_key', 'rt_status', 'error']
print(summary[cols].to_string(index=False))

n_ok    = int((summary['error'].isna() & (summary['n_keys_written'] > 0)).sum())
n_no_rt = int((summary['error'].isna() & (summary['n_rt_finite'] == 0)).sum())
n_err   = int(summary['error'].notna().sum())
print(f"\n  sessions with regenerated pop_mat+RT pairs:  {n_ok}/{len(summary)}")
print(f"  sessions where RT was empty/missing:      {n_no_rt}")
print(f"  sessions that hit a hard error:           {n_err}")

 10%|▉         | 108675/1126560 [00:30<04:37, 3672.34it/s]/home/AD/jugorman/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



=== Per-session summary ===
 bird                      rec  n_units  n_trials_loaded  n_rt_finite  n_keys_written    join_key                                                                               rt_status               error
B1188 2020-11-03_11-28-09_2250       36             1916         1909              32    trial_id                                  loaded 1916 rows, joining on trial_id, 1909 finite RTs                None
B1188 2020-11-04_09-17-36_2250       44              806          806              32 frame_begin            loaded 806 rows, joining on frame_begin (no trial_id column), 806 finite RTs                None
B1188 2020-11-04_18-10-13_2250       14              409            0               4        None      empty trial_events file: exp1_rec1_dat2020-11-04_18-10-13_2250_trial_events.pickle                None
B1188 2020-11-05_08-50-36_2250       35              193            0              28 frame_begin          loaded 3248 rows, joining on frame_begin

In [9]:
for bird, rec, sorter in birds_list:
    ed = find_existing_exp_dir(base_dir_pop_mat, bird, rec)
    if ed is None:
        print(f"{bird}/{rec}: NO exp_dir"); continue
    name = os.path.basename(ed)
    p = find_trial_events_path(trial_events_dir, name)
    rt_series, join_key, msg = load_rt_for_recording(name, trial_events_dir)
    nfin = int(rt_series.notna().sum()) if rt_series is not None else 0
    print(f"{name[:40]:40s} file={'Y' if p else 'N'} key={join_key} finite={nfin} | {msg}")

exp1_rec1_dat2020-11-03_11-28-09_2250    file=Y key=trial_id finite=1909 | loaded 1916 rows, joining on trial_id, 1909 finite RTs
exp1_rec1_dat2020-11-04_09-17-36_2250    file=Y key=frame_begin finite=806 | loaded 806 rows, joining on frame_begin (no trial_id column), 806 finite RTs
exp1_rec1_dat2020-11-04_18-10-13_2250    file=Y key=None finite=0 | empty trial_events file: exp1_rec1_dat2020-11-04_18-10-13_2250_trial_events.pickle
exp1_rec1_dat2020-11-05_08-50-36_2250    file=Y key=frame_begin finite=3231 | loaded 3248 rows, joining on frame_begin (no trial_id column), 3231 finite RTs
exp1_rec1_dat2020-11-05_08-50-36_2250    file=Y key=frame_begin finite=3231 | loaded 3248 rows, joining on frame_begin (no trial_id column), 3231 finite RTs
exp1_rec1_dat2020-11-05_17-51-09_2250    file=Y key=frame_begin finite=155 | loaded 157 rows, joining on frame_begin (no trial_id column), 155 finite RTs
exp1_rec1_dat2020-11-06_08-32-56_2250    file=Y key=frame_begin finite=861 | loaded 863 rows, joi

In [10]:
print(os.path)

<module 'posixpath' (frozen)>


## 7) Verification: shapes match per condition

In [11]:
ok_count = 0
reasons = Counter()
for bird, rec, _ in birds_list:
    exp_dir = find_existing_exp_dir(base_dir_pop_mat, bird, rec)
    if exp_dir is None:
        continue
    for pf in sorted(f for f in os.listdir(exp_dir)
                     if f.endswith('.pkl') and not f.endswith('_rt.pkl')):
        pop_mat_p = os.path.join(exp_dir, pf)
        rt_p   = os.path.join(exp_dir, pf[:-len('.pkl')] + '_rt.pkl')
        if not os.path.exists(rt_p):
            reasons['rt missing'] += 1
            continue
        try:
            Z  = pickle.load(open(pop_mat_p, 'rb'))
            rt = pickle.load(open(rt_p,  'rb'))
        except Exception:
            reasons['other'] += 1
            continue
        n_pop_mat = Z.shape[0] if hasattr(Z, 'shape') else None
        n_rt   = len(rt)    if hasattr(rt, '__len__') else None
        if n_pop_mat is None or n_rt is None or n_pop_mat != n_rt:
            reasons['other'] += 1
        else:
            ok_count += 1

print(f"matching (pop_mat, RT) pairs: {ok_count}")
print(f"mismatches:                {sum(reasons.values())}")
print(f"  - RT missing:            {reasons['rt missing']}")
print(f"  - other (shape/load):    {reasons['other']}")

matching (pop_mat, RT) pairs: 33107
mismatches:                2157
  - RT missing:            2157
  - other (shape/load):    0


In [12]:
import pickle, numpy as np, pandas as pd
from pathlib import Path

rows = []
for bird_dir in sorted(p for p in Path(POP_MAT_OUT).iterdir() if p.is_dir()):
    for exp_dir in sorted(p for p in bird_dir.iterdir() if p.is_dir()):
        conds = sorted(p for p in exp_dir.glob('*.pkl') if not p.name.endswith('_rt.pkl'))
        if not conds:
            continue

        n_units = n_bins = None
        for c in conds:                      # one Z is enough for the shape
            try:
                Z = np.asarray(pickle.load(open(c, 'rb')))
                if Z.ndim == 3:
                    n_units, n_bins = int(Z.shape[1]), int(Z.shape[2])
                    break
            except Exception:
                pass

        n_act = n_pas = n_fin = n_missing = 0
        interps = set()
        for c in conds:
            stem = c.stem
            if '_Z_' in stem:
                interps.add(stem.split('_Z_')[0].split('_', 1)[1])
            rtp = c.with_name(stem + '_rt.pkl')
            if not rtp.exists():
                n_missing += 1
                continue
            rt = np.asarray(pickle.load(open(rtp, 'rb')), dtype=float)
            if   stem.endswith('_active'):  n_act += len(rt)
            elif stem.endswith('_passive'): n_pas += len(rt)
            n_fin += int(np.isfinite(rt).sum())

        _, join_key, msg = load_rt_for_recording(exp_dir.name, trial_events_dir)
        m = _ts_re_cfg.search(exp_dir.name)

        rows.append(dict(
            bird=bird_dir.name, exp_dir=exp_dir.name,
            timestamp=m.group(1) if m else '',
            n_condition_files=len(conds), n_units=n_units, n_bins=n_bins,
            n_interps=len(interps), interps='|'.join(sorted(interps)),
            n_trials_active=n_act, n_trials_passive=n_pas,
            n_trials_total=n_act + n_pas, n_rt_finite=n_fin,
            rt_coverage=round(n_fin / (n_act + n_pas), 4) if (n_act + n_pas) else None,
            n_missing_rt_files=n_missing,
            join_key=join_key or '', rt_status=msg,
        ))

sessions = pd.DataFrame(rows)
out_path = Path(DATA_ROOT) / 'sessions.csv'
sessions.to_csv(out_path, index=False)
print(f"wrote {out_path}  ({len(sessions)} sessions, {sessions.bird.nunique()} birds)")
print(f"  total trials {sessions.n_trials_total.sum():,}, "
      f"finite RT {sessions.n_rt_finite.sum():,}")
print(f"  join keys: {sessions.join_key.value_counts().to_dict()}")
sessions

wrote /mnt/cube/jugorman/EDNpopdyn/data/sessions.csv  (351 sessions, 9 birds)
  total trials 1,062,798, finite RT 167,061
  join keys: {'': 125, 'frame_begin': 116, 'trial_id': 110}


,bird,exp_dir,timestamp,n_condition_files,n_units,n_bins,n_interps,interps,n_trials_active,n_trials_passive,n_trials_total,n_rt_finite,rt_coverage,n_missing_rt_files,join_key,rt_status
0,B1170,exp1_rec1_dat2021-04-08_18-57-54_800,2021-04-08_18-57-54_800,12,46,60,3,AE|BF|CG,0,4079,4079,0,0.0000,0,,empty trial_events file: exp1_rec1_dat2021-04-...
1,B1170,exp1_rec1_dat2021-04-09_08-07-32_800,2021-04-09_08-07-32_800,52,26,60,9,AE|AF|AG|BE|BF|BG|CE|CF|CG,0,430,430,0,0.0000,0,,empty trial_events file: exp1_rec1_dat2021-04-...
2,B1170,exp1_rec1_dat2021-04-09_11-56-08_800,2021-04-09_11-56-08_800,53,65,60,9,AE|AF|AG|BE|BF|BG|CE|CF|CG,1,5890,5891,1,0.0002,0,trial_id,"loaded 6 rows, joining on trial_id, 1 finite RTs"
3,B1170,exp1_rec1_dat2021-04-09_23-19-54_800,2021-04-09_23-19-54_800,22,95,60,1,AE,250,0,250,250,1.0000,0,trial_id,"loaded 266 rows, joining on trial_id, 261 fini..."
4,B1170,exp1_rec1_dat2021-04-10_08-04-30_800,2021-04-10_08-04-30_800,76,70,60,9,AE|AF|AG|BE|BF|BG|CE|CF|CG,964,5895,6859,964,0.1405,0,trial_id,"loaded 1073 rows, joining on trial_id, 1037 fi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,B1595,exp1_rec1_dat2021-04-02_08-54-57_900,2021-04-02_08-54-57_900,172,20,60,9,AE|AF|AG|BE|BF|BG|CE|CF|CG,1775,0,1775,1775,1.0000,0,frame_begin,"loaded 1919 rows, joining on frame_begin (no t..."
347,B1595,exp1_rec1_dat2021-04-02_16-59-43_900,2021-04-02_16-59-43_900,36,21,60,9,AE|AF|AG|BE|BF|BG|CE|CF|CG,0,5981,5981,0,0.0000,0,,empty trial_events file: exp1_rec1_dat2021-04-...
348,B1595,exp1_rec1_dat2021-04-02_22-29-19_900,2021-04-02_22-29-19_900,81,16,60,4,AE|AF|BE|BF,849,0,849,849,1.0000,0,frame_begin,"loaded 904 rows, joining on frame_begin (no tr..."
349,B1595,exp1_rec1_dat2021-04-03_13-29-57_1480,2021-04-03_13-29-57_1480,12,20,60,3,AE|BF|CG,0,10145,10145,0,0.0000,0,,empty trial_events file: exp1_rec1_dat2021-04-...
